# The soft core — taming the Lennard-Jones singularity in MD

*Utrecht University Molecular Modelling courses from the [Bonvin lab](https://bonvinlab.org).*

The molecular-dynamics notebook `LJ-ELEC_MD-VelocityVerlet.ipynb` quietly relies on a
**soft core** to survive long runs. This notebook explains what that is, *why* it is
needed, and shows it in action.

### The problem: the Lennard-Jones wall is infinitely steep

The van der Waals (Lennard-Jones) energy between two particles is

$$U_\text{LJ}(r)=\epsilon\left[\left(\frac{r_\text{min}}{r}\right)^{12}-\left(\frac{r_\text{min}}{r}\right)^{6}\right],$$

a shallow attractive well of depth $\tfrac14\epsilon$ around $r\approx1.12\,r_\text{min}$,
and a **repulsive core that rises like $r^{-12}$** as $r\to0$. The force $-\mathrm{d}U/\mathrm{d}r$
is even steeper, $\sim r^{-13}$ — it **diverges** at contact.

MD advances the particles in finite time steps $h$. As long as particles stay near
the well the steps are small and everything is fine. But if two particles are driven
into hard contact — a fast head-on collision, or (in our charged system) an
oppositely-charged pair pulled together by Coulomb attraction — a single step can land
*deep inside the core*, where the force is astronomically large. The next step then
flings the particle away at enormous speed and the energy **explodes**. Crucially this
is **not fixed by shrinking $h$**: the collision is under-resolved at any practical
time step.

### The fix: cap the distance in the energy/force evaluation

A **soft core** removes the singularity *for the purpose of the simulation*. Below a
chosen separation `SoftCore`, we clamp the pair distance used in the energy and force:

```python
distsquare = max(distsquare, SoftCore**2)
```

A deeply overlapping pair then feels the large-but-**finite** values
$U_\text{LJ}(\text{SoftCore})$ and $F_\text{LJ}(\text{SoftCore})$ instead of infinity.
The integrator stays stable and a would-be explosion becomes a recoverable bounce.
The price is small: the clamp does a little work when it fires (so it is not perfectly
energy-conserving) and it alters the physics at very short range — where the particles
should not be sitting anyway. Real MD codes use the same idea (soft-core potentials),
along with small enough time steps and shifted/truncated potentials.

Only stdlib `math`/`random` plus **matplotlib** are needed.

## 1. Imports

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("matplotlib") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)

import math
import random
import statistics
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.animation import FuncAnimation
rc('animation', html='jshtml', embed_limit=64)
%matplotlib inline

## 2. Energy, force and integrator (with a soft-core switch)

Exactly the Lennard-Jones + Coulomb energy/force and velocity-Verlet + Berendsen
engine of the MD notebook, except that `Calc_Ene2` and `Calc_Force` take a
`softcore2` argument. Passing `softcore2 = SoftCore**2` clamps `distsquare` to it
(soft core **on**); passing `0.0` leaves the bare, singular Lennard-Jones (soft core
**off**) — that is the single switch we compare below.

In [ ]:
def SignR(a, b):
    return a if b > 0 else -a

def dist(A, B):
    return math.sqrt((A[0]-B[0])**2 + (A[1]-B[1])**2)

def charge_color(charge, qat):
    return "#FFFFFF" if charge == qat else "#333333"

# --- Lennard-Jones + Coulomb from the squared distance ---
def LJ2(distsquare, epsilon, rmin_exp6):
    Z = (1 / distsquare) ** 3 * rmin_exp6
    return epsilon * Z * (Z - 1)

def Coulomb2(distsquare, dielec, qa, qb):
    return qa * qb / (dielec * math.sqrt(distsquare))

def Calc_Ene2(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, softcore2=0.0, elec=1):
    Ene = ELJ = ECoul = 0.0
    rmin_exp6 = rmin ** 6
    for i in range(len(coord) - 1):
        for j in range(i + 1, len(coord)):
            d2 = 0.0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                hb = boxdim[k] / 2
                tmp = tmp - SignR(hb, tmp - hb) - SignR(hb, tmp + hb)
                d2 += tmp ** 2
            if d2 < cutoffsquare:
                d2 = max(d2, softcore2)               # <-- soft core
                vdw = LJ2(d2, epsilon, rmin_exp6); Ene += vdw; ELJ += vdw
                if elec:
                    cc = Coulomb2(d2, dielec, coord[i][2], coord[j][2]); Ene += cc; ECoul += cc
    return Ene, ELJ, ECoul

def ForceLJ2(distsquare, epsilon, rmin_exp6, xi):
    rij = math.sqrt(distsquare)
    Z = (1 / distsquare) ** 3 * rmin_exp6
    return epsilon * (2 * Z - 1) * (rmin_exp6 * (-6.0 / rij ** 7)) * (xi / rij)

def ForceCoulomb(distsquare, dielec, qa, qb, xi):
    rij = math.sqrt(distsquare)
    return -1.0 * (qa * qb / dielec) * (1 / distsquare) * (xi / rij)

def Calc_Force(coord, epsilon, rmin, dielec, cutoffsquare, boxdim, softcore2=0.0):
    Force = []; rmin_exp6 = rmin ** 6
    for i in range(len(coord)):
        fx = fy = 0.0
        for j in range(len(coord)):
            if i == j:
                continue
            d2 = 0.0
            for k in range(2):
                tmp = coord[j][k] - coord[i][k]
                hb = boxdim[k] / 2
                tmp = tmp - SignR(hb, tmp - hb) - SignR(hb, tmp + hb)
                d2 += tmp ** 2
            if d2 < cutoffsquare:
                d2 = max(d2, softcore2)               # <-- soft core
                qa, qb = coord[i][2], coord[j][2]
                for k in range(2):
                    tmp = coord[j][k] - coord[i][k]
                    ff = ForceLJ2(d2, epsilon, rmin_exp6, tmp) + ForceCoulomb(d2, dielec, qa, qb, tmp)
                    if k == 0:
                        fx += ff
                    else:
                        fy += ff
        Force.append([fx, fy])
    return Force

def calc_temp(vel, nat, k, mass):
    v2 = sum(vx*vx + vy*vy for vx, vy in vel)
    kin = 0.5 * mass * v2
    return kin, kin / (nat * k)

# --- velocity-Verlet + Berendsen thermostat ---
def VelVerlet_Pos(coord, vel, force, h, mass):
    return [[coord[i][0] + vel[i][0]*h + 0.5*force[i][0]/mass*h**2,
             coord[i][1] + vel[i][1]*h + 0.5*force[i][1]/mass*h**2,
             coord[i][2]] for i in range(len(coord))]

def VelVerlet_Vel(vel, f_old, f_new, h, mass):
    return [[vel[i][0] + 0.5*(f_old[i][0] + f_new[i][0])/mass*h,
             vel[i][1] + 0.5*(f_old[i][1] + f_new[i][1])/mass*h] for i in range(len(vel))]

def Berendsen(vel, Tcur, T0, h, tau):
    if Tcur <= 0.0:
        return vel
    lam = math.sqrt(max(1.0 + (h/tau)*(T0/Tcur - 1.0), 0.0))
    return [[v[0]*lam, v[1]*lam] for v in vel]

## 3. Parameters

The same 20-particle LJ + Coulomb system as the MD notebook. `SoftCore = 0.8 * Rmin`
is the clamp distance; everything else is the standard shared set.

In [ ]:
nAtoms  = 20
Radius  = 25.0
Mass    = 10.0
Rmin    = 2.24 * Radius
BoxDim  = [500.0, 500.0]
Epsilon = 25.0
Dielec  = 1.0
qat     = Radius
frac_neg = 0.5
OverlapFr = 0.0
CutOff  = 250.0
CutOffSquare = CutOff ** 2
cstboltz = 1000 * 0.00198722 / 4.18
Seed    = 100

SoftCore  = 0.8 * Rmin       # soft-core distance
SoftCore2 = SoftCore ** 2

Temperature = 300.0          # thermostat target
timestep    = 1.0e-3
tauT        = 0.1
nsteps      = 40000          # long enough that the no-soft-core run collides (~step 34000)
print(f"Rmin = {Rmin:.1f}   SoftCore = 0.8*Rmin = {SoftCore:.1f}   "
      f"LJ well minimum at r = {1.122*Rmin:.1f}")

## 4. The van der Waals function, with and without the soft core

Below is the Lennard-Jones **energy** and **radial force** as a function of the pair
separation $r$. The dashed grey curve is the true LJ — it shoots to $+\infty$ as
$r\to0$. The solid blue curve is the soft-cored version: for $r<$ `SoftCore` the
distance is clamped, so the energy and force **flatten to a finite plateau**
($U_\text{LJ}(\text{SoftCore})$ and $F_\text{LJ}(\text{SoftCore})$) instead of
blowing up. Above `SoftCore` the two are identical — the attractive well and the
whole physically-relevant range are untouched.

**Reading the curve near $r\approx r_\text{min}=56$.** The LJ energy is *repulsion
minus attraction*, $\epsilon[(r_\text{min}/r)^{12}-(r_\text{min}/r)^{6}]$, and only
follows the bare $r^{-12}$ core (dotted green) at *small* $r$. Near $r_\text{min}$
the two terms nearly cancel (at $r=55$: $+31$ repulsion, $-28$ attraction, net $+3$),
so the function crosses **zero exactly at $r=r_\text{min}=56$** and dips into the
attractive well (minimum $-\tfrac14\epsilon$ at $1.12\,r_\text{min}=62.8$). That
attraction↔repulsion crossover is the gentle "bend" near $r=55$ — a real feature of
the LJ *well*, not the soft core; the curve is perfectly smooth. (The $y$-axis is
simply clipped, so the true LJ runs off the top as it diverges toward $+\infty$.)

In [ ]:
rmin6 = Rmin ** 6

def U_lj(r, softcore=0.0):
    d2 = max(r*r, softcore*softcore)
    return LJ2(d2, Epsilon, rmin6)

def F_lj(r, softcore=0.0):
    rr = max(r, softcore)                 # clamp
    Z = rmin6 / rr**6                     # (rmin/rr)^6
    return 6 * Epsilon * Z * (2*Z - 1) / rr   # radial force = -dU/dr  (+ve = repulsive)

rs = [0.55*Rmin + i*(1.6*Rmin)/400 for i in range(401)]   # r from 0.55 to 2.15 * Rmin

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
# energy
ax[0].plot(rs, [Epsilon*(Rmin/r)**12 for r in rs], color="#2ca02c", ls=":", lw=1.4,
           label=r"repulsion only  $\epsilon(r_{min}/r)^{12}$")
ax[0].plot(rs, [U_lj(r, 0.0)      for r in rs], color="#999999", ls="--", lw=1.6, label="true LJ (singular)")
ax[0].plot(rs, [U_lj(r, SoftCore) for r in rs], color="#1f77b4", lw=2.0, label="with soft core")
ax[0].axvline(SoftCore, color="#d62728", ls=":", lw=1.2)
ax[0].text(SoftCore, 1.5, " SoftCore", color="#d62728", fontsize=8, rotation=90, va="bottom")
ax[0].axvline(Rmin, color="#8c564b", ls="-.", lw=0.9)
ax[0].text(Rmin, -7, r" $r_{min}$ (U=0)", color="#8c564b", fontsize=8, va="top")
ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_ylim(-30, 350)     # clipped: the true LJ runs off the top as it diverges
ax[0].set_xlabel("pair separation r"); ax[0].set_ylabel("van der Waals energy")
ax[0].set_title("LJ energy: the soft core caps the $r^{-12}$ wall"); ax[0].legend(fontsize=8)
# force
ax[1].plot(rs, [F_lj(r, 0.0)      for r in rs], color="#999999", ls="--", lw=1.6, label="true LJ (singular)")
ax[1].plot(rs, [F_lj(r, SoftCore) for r in rs], color="#1f77b4", lw=2.0, label="with soft core")
ax[1].axvline(SoftCore, color="#d62728", ls=":", lw=1.2)
ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_ylim(-60, 300)     # clipped: the true force runs off the top as it diverges
ax[1].set_xlabel("pair separation r"); ax[1].set_ylabel("radial force  (+ = repulsive)")
ax[1].set_title("LJ force: capped instead of diverging"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"at r = SoftCore = {SoftCore:.1f}:  U_LJ = {U_lj(SoftCore):.0f}   "
      f"F_LJ = {F_lj(SoftCore):.0f}   (finite — below this the soft core holds these values)")
print(f"without the soft core, at r = 0.5*Rmin = {0.5*Rmin:.0f}:  "
      f"U_LJ = {U_lj(0.5*Rmin):.0f}   F_LJ = {F_lj(0.5*Rmin):.0f}   (and → ∞ as r → 0)")

## 5. MD with and without the soft core

Now the point of it all. We run the **same** 20-particle system twice with the
velocity-Verlet integrator and the Berendsen thermostat — once with the soft core on,
once off — from the identical start, and record the total energy, temperature and the
**closest pair distance** at every step.

Watch what happens around $t\approx34$: an oppositely-charged pair reaches hard
contact. **Without** the soft core the closest distance collapses, the force
diverges, and the energy and temperature blow up (the run is stopped when it
detects $T>10^5$). **With** the soft core the same encounter is a harmless bounce —
the closest distance is held near `SoftCore`, and the run sails on to the end.

In [ ]:
def InitConf(n, dim, radius, qat, frac_neg, seed):
    random.seed(seed)
    coord = []; nneg = int(n * frac_neg)
    def place(charge):
        while True:
            x = random.random() * (dim[0] - 2*radius) + radius
            y = random.random() * (dim[1] - 2*radius) + radius
            if all(dist(c, [x, y]) >= (1 - OverlapFr) * 2 * radius for c in coord):
                coord.append([x, y, charge]); return
    for _ in range(nneg):     place(-qat)
    for _ in range(n - nneg): place(+qat)
    return coord

def InitVel(n, temperature, cstboltz, mass):
    # continues the RNG stream seeded by InitConf (like the GUI script) so this run
    # reproduces exactly the hard collision that motivated the soft core (~t=34)
    stdev = math.sqrt(cstboltz * temperature / mass)
    v = []
    for _ in range(n):
        r1, r2 = random.random(), random.random()
        v.append([math.sqrt(-2*math.log(r1))*math.cos(r2)*stdev,
                  math.sqrt(-2*math.log(r1))*math.sin(0.5*r2)*stdev])
    vx = sum(a[0] for a in v)/n; vy = sum(a[1] for a in v)/n
    for a in v: a[0] -= vx; a[1] -= vy
    _, tt = calc_temp(v, n, cstboltz, mass); sc = math.sqrt(temperature / tt)
    return [[a[0]*sc, a[1]*sc] for a in v]

def min_pair_dist(coord):
    best = 1e9
    for i in range(len(coord) - 1):
        for j in range(i + 1, len(coord)):
            d2 = 0.0
            for k in range(2):
                t = coord[j][k] - coord[i][k]; hb = BoxDim[k]/2
                t = t - SignR(hb, t-hb) - SignR(hb, t+hb); d2 += t*t
            best = min(best, d2)
    return math.sqrt(best)

def run_md(softcore2, nsteps, dt):
    coord = InitConf(nAtoms, BoxDim, Radius, qat, frac_neg, Seed)
    vel   = InitVel(nAtoms, Temperature, cstboltz, Mass)
    Force = Calc_Force(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim, softcore2)
    Etot, Temp, Dmin, traj = [], [], [], []
    exploded = None
    for it in range(nsteps):
        coord = VelVerlet_Pos(coord, vel, Force, dt, Mass)
        Fnew = Calc_Force(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim, softcore2)
        vel = VelVerlet_Vel(vel, Force, Fnew, dt, Mass); Force = Fnew
        Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)
        vel = Berendsen(vel, T, Temperature, dt, tauT)
        Kin, T = calc_temp(vel, nAtoms, cstboltz, Mass)
        Ene, _, _ = Calc_Ene2(coord, Epsilon, Rmin, Dielec, CutOffSquare, BoxDim, softcore2)
        for pp in range(nAtoms):
            for i in range(2):
                if coord[pp][i] < 0:         coord[pp][i] += BoxDim[i]
                if coord[pp][i] > BoxDim[i]: coord[pp][i] -= BoxDim[i]
        Etot.append(Ene + Kin); Temp.append(T); Dmin.append(min_pair_dist(coord))
        traj.append([[c[0], c[1], c[2]] for c in coord])
        if T > 1e5:                       # exploded -> stop
            exploded = it; break
    return dict(Etot=Etot, Temp=Temp, Dmin=Dmin, traj=traj, exploded=exploded)

off = run_md(0.0,       nsteps, timestep)   # soft core OFF
on  = run_md(SoftCore2, nsteps, timestep)   # soft core ON
print(f"soft core OFF: {'EXPLODED at step '+str(off['exploded']) if off['exploded'] else 'stable'}")
print(f"soft core ON : {'EXPLODED at step '+str(on['exploded']) if on['exploded'] else 'stable'}"
      f"   <T> = {statistics.mean(on['Temp']):.0f} K")

The three panels below: total energy, temperature, and the closest pair distance
over time. The red (no soft core) curves run off the scale at the collision; the blue
(soft core) curves stay bounded.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
def tt(r): return [i*timestep for i in range(len(r["Etot"]))]

if off["exploded"]: ax[0].axvline(off["exploded"]*timestep, color="#d62728", ls=":", lw=1)
ax[0].plot(tt(off), off["Etot"], color="#d62728", lw=0.8, label="no soft core")
ax[0].plot(tt(on),  on["Etot"],  color="#1f77b4", lw=0.8, label="soft core")
ax[0].set_ylim(1500, 4500)     # clipped: the no-soft-core run spikes off the top when it explodes
ax[0].set_xlabel("time"); ax[0].set_ylabel("total energy"); ax[0].legend(fontsize=8)
ax[0].set_title("Total energy (no-soft-core run explodes off the top)")

if off["exploded"]: ax[1].axvline(off["exploded"]*timestep, color="#d62728", ls=":", lw=1)
ax[1].plot(tt(off), off["Temp"], color="#d62728", lw=0.8, label="no soft core")
ax[1].plot(tt(on),  on["Temp"],  color="#1f77b4", lw=0.8, label="soft core")
ax[1].axhline(Temperature, color="k", ls="--", lw=1)
ax[1].set_ylim(0, 1000)        # clipped: the no-soft-core run spikes off the top
ax[1].set_xlabel("time"); ax[1].set_ylabel("temperature (K)"); ax[1].legend(fontsize=8)
ax[1].set_title("Temperature")

ax[2].plot(tt(off), off["Dmin"], color="#d62728", lw=0.8, label="no soft core")
ax[2].plot(tt(on),  on["Dmin"],  color="#1f77b4", lw=0.8, label="soft core")
ax[2].axhline(SoftCore, color="#d62728", ls=":", lw=1.2, label="SoftCore")
ax[2].set_xlabel("time"); ax[2].set_ylabel("closest pair distance"); ax[2].legend(fontsize=8)
ax[2].set_ylim(0, 2.2*Radius); ax[2].set_title("Closest approach (the collision)")
plt.tight_layout(); plt.show()

## 6. Watch the two simulations

The two boxes side by side (white $=$ positive charge, dark $=$ negative), from the
identical start. They are indistinguishable until $t\approx34$, when an
oppositely-charged pair reaches hard contact. **With the soft core** (left) the pair
bounces and everything stays in the box. **Without it** (right) the colliding pair is
flung out at enormous speed and vanishes off-screen — the simulation has blown up and
is stopped.

In [ ]:
with_sc, without_sc = on["traj"], off["traj"]   # left = soft core, right = none
stride = max(1, len(without_sc) // 120)
frames = list(range(0, len(without_sc), stride))
if frames[-1] != len(without_sc) - 1:
    frames.append(len(without_sc) - 1)              # include the final (exploded) frame

def xy(tr, n):
    n = min(n, len(tr) - 1)
    return [[c[0], c[1]] for c in tr[n]], [charge_color(c[2], qat) for c in tr[n]]

fig, ax = plt.subplots(1, 2, figsize=(10, 5.4))
scats = []
for a, title, tr in ((ax[0], "with soft core", with_sc), (ax[1], "no soft core", without_sc)):
    a.set_xlim(0, BoxDim[0]); a.set_ylim(0, BoxDim[1]); a.set_aspect("equal")
    a.set_facecolor("#ccddff"); a.set_xticks([]); a.set_yticks([]); a.set_title(title)
    pts, cols = xy(tr, 0)
    scats.append(a.scatter([p[0] for p in pts], [p[1] for p in pts], s=340, c=cols, edgecolors="k"))
sup = fig.suptitle("")

def update(n):
    for s, tr in zip(scats, (with_sc, without_sc)):
        pts, cols = xy(tr, n)
        s.set_offsets(pts); s.set_color(cols)
    sup.set_text(f"step {n}   t = {n*timestep:.1f}")
    return scats

anim = FuncAnimation(fig, update, frames=frames, interval=80, blit=False)
plt.close(fig); anim

## 7. Take-home messages

- The Lennard-Jones repulsive core rises like $r^{-12}$ (force $\sim r^{-13}$) and is
  **singular** at contact. Fine near the well, fatal in a hard collision.
- With a finite time step the integrator cannot resolve a deep-core encounter and the
  energy **explodes**. Shrinking the step barely helps — the collision happens at the
  same instant regardless (see the MD notebook's diagnosis).
- A **soft core** clamps the pair distance, `distsquare = max(distsquare, SoftCore**2)`,
  so the energy and force stay **finite** below `SoftCore`. The dangerous collision
  becomes a bounce; the run stays stable (§5).
- It is a pragmatic modification, not free physics: the clamp does a little work when it
  fires (mildly non-energy-conserving) and changes the potential at very short range.
  Choose `SoftCore` small enough that it only ever triggers on a genuine hard
  collision, not during normal thermal contact (here `0.8 * Rmin`).
- The same idea appears throughout molecular simulation: soft-core potentials for
  free-energy calculations, capped/shifted forces, and simply using a small enough time
  step in the first place.